In [1]:
import gymnasium as gym
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import random

from tensorflow.keras import models, layers, optimizers, losses
from collections import deque

In [2]:
# Definiendo hiperparámetros
EPISODES = 500
BATCH_SIZE = 64
GAMMA = 0.99
LEARNING_RATE = 0.001
BUFFER_SIZE = 10_000
TARGET_UPDATE_FREQ = 10
EPSILON_START = 1.0
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.995

In [3]:
# Definimos la clase DQNAgent
class DQNAgent:
    def __init__(self, state_size: int, action_size: int):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=BUFFER_SIZE)
        self.epsilon = EPSILON_START

        self.model = self._build_model()        # Q(s, a, \theta)
        self.target_model = self._build_model() # Q(s, a, \theta^-)
        self.update_target_network()

        self.optimizer = optimizers.Adam(learning_rate=LEARNING_RATE)
        self.loss = losses.Huber()  # Más robusta a outliers que MSE

    def _build_model(self):
        model = models.Sequential([
            layers.Input(shape=(self.state_size,)),
            layers.Dense(24, activation='relu'),
            layers.Dense(24, activation='relu'),
            layers.Dense(self.action_size, activation='linear')  # Salida: Q-values para cada acción
        ])
        return model

    def update_target_network(self):
        self.target_model.set_weights(self.model.get_weights())

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if np.random.rand() < self.epsilon:
            return random.randrange(self.action_size)  # Acción aleatoria
        q_values = self.model(state, training=False)
        return np.argmax(q_values[0])  # Acción con el mayor Q-value

    def replay(self):
        if len(self.memory) < BATCH_SIZE:
            return  # No hay suficientes experiencias para entrenar
        minibatch = random.sample(self.memory, BATCH_SIZE)

        states = np.vstack([t[0] for t in minibatch])
        actions = np.array([t[1] for t in minibatch])
        rewards = np.array([t[2] for t in minibatch], dtype=np.float32)
        next_states = np.vstack([t[3] for t in minibatch]).astype(np.float32)
        dones = np.array([t[4] for t in minibatch], dtype=np.float32)

        next_q_values = self.target_model(next_states, training=False)
        max_next_q_values = tf.reduce_max(next_q_values, axis=1)

        # Ecuación de Bellman
        targets = rewards + (1 - dones) * GAMMA * max_next_q_values

        with tf.GradientTape() as tape:
            q_values = self.model(states, training=True)
            action_masks = tf.one_hot(actions, self.action_size)
            q_actions = tf.reduce_sum(q_values * action_masks, axis=1)
            loss = self.loss(targets, q_actions)

        grads = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))

        if self.epsilon > EPSILON_MIN:
            self.epsilon *= EPSILON_DECAY

Epsilon greedy es una política / método de exploración utilizado en algoritmos de aprendizaje por refuerzo, como DQN (Deep Q-Network). Consiste en seleccionar una acción aleatoria con una probabilidad epsilon (ε) y seleccionar la acción con el valor Q más alto con una probabilidad de 1 - ε. Esto permite al agente explorar nuevas acciones mientras también aprovecha las acciones que ya ha aprendido que son efectivas. A medida que el entrenamiento avanza, ε se reduce gradualmente para favorecer la explotación sobre la exploración.

In [4]:
env = gym.make('CartPole-v1')
state_size = int(env.observation_space.shape[0])
action_size = int(env.action_space.n)

agent = DQNAgent(state_size, action_size)
epochs_rewards = []

for e in range(EPISODES):
    state, _ = env.reset()
    state = np.reshape(state, [1, state_size])
    total_reward = 0
    
    for time in range(500):
        # Get action
        action = agent.act(state)

        # Take action (obtain r, s', done)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state = np.reshape(next_state, [1, state_size])

        # Modify reward
        reward = reward if not done or time == 499 else -10

        # Store transition in memory
        agent.remember(state, action, reward, next_state, done)

        # State transition
        state = next_state
        total_reward += 1

        agent.replay()  # Train the agent with the experience of the episode

        if done:
            print(f"Episode: {e+1}/{EPISODES}, Reward: {total_reward}")
            break

    epochs_rewards.append(total_reward)

    if (e + 1) % TARGET_UPDATE_FREQ == 0:
        agent.update_target_network()  # Update target network every TARGET_UPDATE_FREQ episodes
        print(f">>>>> Target network updated")

env.close()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_rewards)
ax.set_xlabel("Epochs")
ax.set_ylabel("Reward")
ax.set_title("DQN - CartPole v1")
plt.show()

Episode: 1/500, Reward: 54
Episode: 2/500, Reward: 31
Episode: 3/500, Reward: 36
Episode: 4/500, Reward: 40
Episode: 5/500, Reward: 24
Episode: 6/500, Reward: 14
Episode: 7/500, Reward: 15
Episode: 8/500, Reward: 21
Episode: 9/500, Reward: 52
Episode: 10/500, Reward: 13
>>>>> Target network updated
Episode: 11/500, Reward: 53
Episode: 12/500, Reward: 48
Episode: 13/500, Reward: 39
Episode: 14/500, Reward: 21
Episode: 15/500, Reward: 11
Episode: 16/500, Reward: 15
Episode: 17/500, Reward: 10
Episode: 18/500, Reward: 9
Episode: 19/500, Reward: 42
Episode: 20/500, Reward: 9
>>>>> Target network updated
Episode: 21/500, Reward: 10
Episode: 22/500, Reward: 11
Episode: 23/500, Reward: 13
Episode: 24/500, Reward: 14
Episode: 25/500, Reward: 15
Episode: 26/500, Reward: 12
Episode: 27/500, Reward: 11
Episode: 28/500, Reward: 33
Episode: 29/500, Reward: 9
Episode: 30/500, Reward: 17
>>>>> Target network updated
Episode: 31/500, Reward: 9
Episode: 32/500, Reward: 9
Episode: 33/500, Reward: 10
Epi

KeyboardInterrupt: 